# Task 3 - Kafka topic and event design

Nodes, edges, metadata, and parser failures have separate topics. Every record carries schema version, event time, file content hash, run ID, stable event ID, and an operation.

In [1]:
!docker compose exec -T broker kafka-topics --bootstrap-server broker:29092 --describe

Topic: cpg.parser-errors.v1	TopicId: 0MJGt2teQquWtGdKsh0UUQ	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=delete,retention.ms=604800000
	Topic: cpg.parser-errors.v1	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
Topic: lab04-connect-configs	TopicId: ZSnolUQURnO6M-RqEz-gdw	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=compact
	Topic: lab04-connect-configs	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
Topic: cpg.edges.v1	TopicId: tQC-F1ksTbm5HvcTgilxjQ	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=compact
	Topic: cpg.edges.v1	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
Topic: __transaction_state	TopicId: T01LOmn-TRqvYCky1ZohWg	PartitionCount: 50	ReplicationFactor: 1	Configs: compression.type=uncompressed,min.insync.replicas=1,cleanup.policy=compact,segment.bytes=104857600,unclean.leader.election.enable=false
	Topic: __transaction_state	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	Las

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from cpg_parser.manifest import Manifest
from cpg_parser.publisher import MemoryPublisher
from cpg_parser.service import ParserService

with TemporaryDirectory() as directory:
    publisher = MemoryPublisher()
    with Manifest(Path(directory) / 'manifest.sqlite') as manifest:
        ParserService(Path('../source-repo'), 'huggingface/optimum', publisher, manifest).process('optimum/version.py', force=True)
    samples = {}
    for topic, key, value in publisher.records:
        samples.setdefault(topic, {'key': key, 'value': value})
print(samples)

{
  "cpg.nodes.v1": {
    "key": "b48918aa8c75bba8b82275c2b6b971a89a85434bd0cdcd8d02fc9cd0b59302e8",
    "value": {
      "schema_version": "1.0",
      "event_time": "2026-07-20T06:24:57.637989Z",
      "repo_id": "huggingface/optimum",
      "file_id": "867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7",
      "run_id": "1ff9e49fd38a457d9e0fceb7ee7fe57b",
      "content_hash": "079eca803ea5bfe068bc805997b013cc14c9ddc8629a2ec634d12bcebb6720ce",
      "op": "upsert",
      "node": {
        "id": "b48918aa8c75bba8b82275c2b6b971a89a85434bd0cdcd8d02fc9cd0b59302e8",
        "kind": "AST",
        "ast_type": "Module",
        "file_id": "867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7",
        "structural_path": "$",
        "line": 0,
        "column": 0,
        "end_line": 0,
        "end_column": 0,
        "name": "",
        "value": ""
      },
      "event_id": "63b8fd885ad93687b36df566474246bd19b54e511ca2fa80404964934a87d076"
    }
  },
  "cpg.edg

## Reflection

Compaction is useful for retained state but does not itself prevent duplicate writes. Idempotency therefore also exists in Kafka keys, Neo4j MERGE statements, MongoDB upserts, and the parser manifest.